# 第 7 章 · 规模化的代价：动作越多，PPO 越要堆算力

配套讲义：仓库里 `robot-motion-rl/notes/lessons/07-scaling-cost.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（数字全部来自 SuperMimic §4.1）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 GPU-h | GPU × 小时 | `ratios()` | 21504、3、7168×、56× |
| 第三步 修正 | ÷ (2.31 × 5.5) | `leftover_gap(7168)` | ≈ 564× |
| 第四步 PPO 加卡 | — | `main()` 第三段 | 0.60 / 0.70 / 0.72 vs 1.2 h 到 0.82 |

数字来自 SuperMimic 论文 §4.1 "Compute accounting"（supermimic-2610.28480.txt L329–L337）。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `gpu_hours`**：GPU 时 = GPU 数 × 小时数。

In [ ]:
def gpu_hours(gpus, hours):
    """GPU 时 = GPU 数 × 小时数。"""
    return gpus * hours

**函数 `ratios`**：返回 (SONIC GPU 时, SuperMimic GPU 时, GPU 时比, 墙钟比)。

In [ ]:
def ratios(sonic=(128, 168), supermimic=(1, 3)):
    """返回 (SONIC GPU 时, SuperMimic GPU 时, GPU 时比, 墙钟比)。"""
    a, b = gpu_hours(*sonic), gpu_hours(*supermimic)
    return a, b, a / b, sonic[1] / supermimic[1]

**函数 `leftover_gap`**：把数据量差（2.3×）和硬件峰值差（≤ 5.5×）都算给 SONIC 之后，剩下的倍数。

In [ ]:
def leftover_gap(ratio, data_factor=611 / 264.56, hw_factor=5.5):
    """把数据量差（2.3×）和硬件峰值差（≤ 5.5×）都算给 SONIC 之后，剩下的倍数。"""
    return ratio / (data_factor * hw_factor)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 GPU 时")
a, b, r, w = ratios()
print(f"SONIC 128 × 168 = {a} GPU 时；SuperMimic 1 × 3 = {b} GPU 时")
print(f"GPU 时比 ≈ {r:.0f}×，墙钟比 = {w:.0f}×")

In [ ]:
print("\n第二步 扣掉数据和硬件")
print(f"数据 611 / 264.56 = {611 / 264.56:.2f}×，硬件峰值最多 5.5× → 合计 {611 / 264.56 * 5.5:.1f}×")
print(f"剩下的差距 ≈ {leftover_gap(r):.0f}×")

In [ ]:
print("\n第三步 PPO 加卡")
for g, ret in [(1, 0.60), (4, 0.70), (8, 0.72)]:
    print(f"PPO-Sim {g} GPU，18 h 后回报 {ret}")
print("SuperMimic 1 GPU，1.2 h 到 0.82")

## 3. 代码里才看得到的

- 论文只写「超过 1000×」「超过 50×」，是保守的下界；代码给的是精确比值 7168 和 56。讲义按精确值算，按论文的保守说法下结论。

## 4. 自己改一改

- 把 SuperMimic 换成 2 卡 9.8 小时，重算三个倍数。
- 只扣数据差或只扣硬件差，看哪一项影响大。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 扩动作两条路：专家蒸馏有损，通用策略烧卡；
> 一百二十八卡七天两万一，一卡三时，差七千；
> 扣掉数据二点三、硬件五点五，还剩五百多；
> 同一副身体，PPO 每条重摸一遍，加卡不变聪明。